In [ ]:
X_test_aligned = test[train_cols].copy()
X_test_scaled_array = scaler.transform(X_test_aligned[scaled_cols])

# combining scale and unscaled data
unscaled_cols = [c for c in train_cols if c not in scaled_cols]
X_test_input = np.hstack([X_test_scaled_array, X_test_aligned[unscaled_cols].values])

# Defining the DNN model.
INPUT_DIM = X_test_input.shape[1]
model = Sequential([
    Dense(256, activation='relu', input_dim=INPUT_DIM),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='linear')
])

model.compile(optimizer=Adam(0.001), loss='mse', metrics=[rmspe])

# loading the trained weights
model.load_weights("dnn.weights.h5")

# Predict on the test set
y_pred_log = model.predict(X_test_input, verbose=0)
y_pred = np.expm1(y_pred_log)

# Fixing negative predictions 
y_pred = np.where(np.isfinite(y_pred), y_pred, 0)  
y_pred = np.clip(y_pred, 0, None)  

# submisison dataset
submission = pd.DataFrame({
    "Id": test_ids.astype(int),
    "Sales": y_pred.flatten().astype(float)
})

# saving csv to kaggle
submission.to_csv("submission_final_dnn.csv", index=False, encoding='utf-8')



/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/var/folders/n1/w1_yvs_x57l_7v6xy03xjqkm0000gn/T/ipykernel_1912/1061803829.py:62: RuntimeWarning: overflow encountered in expm1
  y_pred = np.expm1(y_pred_log)


In [ ]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# RMSPE metric
def rmspe(y_true, y_pred):
    y_true_exp = tf.math.expm1(y_true)
    y_pred_exp = tf.math.expm1(y_pred)
    pct_error = (y_true_exp - y_pred_exp) / K.clip(y_true_exp, K.epsilon(), None)
    return K.sqrt(K.mean(K.square(pct_error)))

# Loading datasets
train = pd.read_csv("train_cleaned.csv") 
test = pd.read_csv("test_cleaned.csv")
test_ids = test["Id"]

# Load scalers and training column info
with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("train_cols.pkl", "rb") as f:
    train_cols = pickle.load(f)

with open("scaler_cols.pkl", "rb") as f:
    scaled_cols = pickle.load(f)  


for col in train_cols:
    if col not in test.columns:
        test[col] = 0.0
X_test_aligned = test[train_cols].copy()

# Scaling numeric/sequence columns ---
X_test_scaled_array = scaler.transform(X_test_aligned[scaled_cols])

# Combining scaled and unscaled columns
unscaled_cols = [c for c in train_cols if c not in scaled_cols]
X_test_input = np.hstack([X_test_scaled_array, X_test_aligned[unscaled_cols].values])

# model
INPUT_DIM = X_test_input.shape[1]
model = Sequential([
    Dense(256, activation='relu', input_dim=INPUT_DIM),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='linear')
])

model.compile(optimizer=Adam(0.001), loss='mse', metrics=[rmspe])

#Loading trained DNN weights
model.load_weights("dnn.weights.h5")

# Predicting on test set
y_pred_log = model.predict(X_test_input, verbose=0)
y_pred = np.expm1(y_pred_log)

# Fix non-finite and negative predictions
y_pred = np.where(np.isfinite(y_pred), y_pred, 0)
y_pred = np.clip(y_pred, 0, None)  

# saving submission file
submission = pd.DataFrame({
    "Id": test_ids.astype(int),
    "Sales": y_pred.flatten().astype(float)
})

submission.to_csv("sub_dnn.csv", index=False, encoding='utf-8')



/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


🎉 Submission ready and saved as 'sub_dnn.csv'!


/var/folders/n1/w1_yvs_x57l_7v6xy03xjqkm0000gn/T/ipykernel_1912/895403293.py:65: RuntimeWarning: overflow encountered in expm1
  y_pred = np.expm1(y_pred_log)
